
# C7-cnn-transfer — Review

Work through this notebook *after* the three lesson sessions and
(ideally) the practice sets.
It is a consolidation tool: a concept summary table, the conv/ResNet
idiom sheet, a 14-item self-quiz, and pointers on what to redo.
Quiz answers are collapsed at the very end — commit to your answers
before looking.


> **Float register reminder (no imports needed for this review):** the course
> convention is `torch.set_default_dtype(torch.float64)` — but any notebook
> that loads pretrained `resnet50` runs in the **float32 register** instead
> (the weights are a float32 artifact; cast inputs at the model boundary and
> record anchors from that pipeline).


## Concept summary

| Concept | One-line summary | Key fact to retain |
|---|---|---|
| Convolution | A small kernel slides over the input; each output is the local weighted sum | No flip (torch = cross-correlation); valid length $n - K + 1$; general output size $\lfloor(n + 2p - K)/s\rfloor + 1$ |
| Feature maps | One 2-D response map per kernel in the bank, stacked along channels | `(N, C, H, W)` in, `(N, F, H', W')` out; weight `(F, C, K, K)`; each output channel sums over *all* input channels |
| Receptive field | The input patch one output depends on | $r \leftarrow r + (K-1)J$, then $J \leftarrow Js$; stride-1 stacks: $r = 1 + \sum (K_\ell - 1)$; check = impulse through all-ones kernels |
| Feature hierarchy | edges → textures → parts → objects as depth grows | Early: small RF, dense, rough maps; late: large RF, sparse, smooth; grids shrink while channels grow ($56^2{\times}256 \to 7^2{\times}2048$) |
| ResNet-50 anatomy | stem (`conv1,bn1,relu,maxpool`), `layer1..4`, `avgpool`, `fc` | Blocks per stage `[3,4,6,3]`; $3{\times}16 + 2 = 50$; stage outputs $256/512/1024/2048$ channels at $56/28/14/7$; `flatten` lives in `forward`, not in `children()` |
| Bottleneck block | 1×1 reduce → 3×3 → 1×1 expand (+skip), expansion 4 | Interior convs: $4m^2 + 9m^2 + 4m^2 = 17m^2$; BN $2C$ per layer ($12m$ per block); stage openers add a downsample (1×1, stride 2 except `layer1`) |
| Model truncation | `nn.Sequential(*list(model.children())[:k])`, optionally `+ layerN[:j]` | List `children()` once (generator!); slices share modules (no copies); cuts respect block boundaries |
| Layer freezing | `p.requires_grad = False` in a loop; prefix-filter `named_parameters()` for selective freezes | Flags record intent, change no outputs; audit in two currencies — tensors (`sum(1 for ...)`) and scalars (`sum(p.numel() ...)`) |
| Transfer learning | Frozen pretrained backbone + pool/flatten + fresh `nn.Linear(f, k)` head | Audit signature: output `(B, k)`; trainable = exactly the head ($f k + k$; full-depth surgery: $2049k$); freeze *before* grafting |



## Idiom sheet

**Reproducible load (Session 2 §1)** — cache header first, then:

```python
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
model.eval()                       # BatchNorm -> stored statistics
with torch.inference_mode():       # every forward, always
    y = model(x.to(torch.float32)) # cast at the model boundary
```

(The cache header — repo-root `TORCH_HOME` resolution — is the first
cell of every weights-loading notebook in this unit.)

**Reading (Session 2):**

```python
[n for n, _ in model.named_children()]           # the ten pieces
len(model.layer3)                                # blocks in a stage
model.layer2[2]                                  # one block, printed anatomy
sum(p.numel() for p in m.parameters())           # scalar count (as the CHECK)
```

**Counting (hand register, Session 2 §5):** conv = out·in·K²
(ResNet convs have no bias); BN = 2C (buffers are not parameters);
interior bottleneck convs $17m^2$, +BN $12m$; dense head
out·in + out; full-depth head swap: $2049k$.
Under a "no `numel`" ban, `torchsummary`, `state_dict` size reads,
`p.nelement()`, and `np.prod(p.shape)` are the same helper in
disguise and score zero.

**Truncation + freezing + transfer (Session 3):**

```python
ch = list(model.children())                      # list ONCE
trunk = nn.Sequential(*ch[:6])                   # through layer2
mid   = nn.Sequential(*ch[:5], model.layer2[:2]) # mid-stage cut
for p in trunk.parameters():                     # freeze at scale
    p.requires_grad = False
head = nn.Linear(512, k)                         # fresh head: flags True by birth
# audits:
all(not p.requires_grad for p in trunk.parameters())
sorted(n for n, p in net.named_parameters() if p.requires_grad)
sum(p.numel() for p in net.parameters() if p.requires_grad)
```

**Errors worth recognizing on sight:**
`mat1 and mat2 shapes cannot be multiplied (...x1 and ...)` (missing
`Flatten` between pool and head);
`Expected 3D (unbatched) or 4D (batched) input to conv2d` (forgot
`(N, C, H, W)`);
`expected scalar type ... but found ...` / `must have the same dtype`
(float32/float64 met at a boundary — cast toward the artifact);
a second pass over `model.children()` yielding nothing (generator
exhausted — list it once).



## Self-quiz (14 items)

Answer everything before opening the collapsed answers at the very
end.

**Q1.** By hand: the full valid output of $x = (1, 2, -1, 0, 3)$
convolved (no flip) with $k = (2, 1)$.

**Q2.** `nn.Conv2d(3, 10, kernel_size=5, stride=2, padding=2)` on a
`(4, 3, 63, 63)` input: output shape, weight shape, and the count of
its parameters with bias included.

**Q3.** State the no-flip convention: what does torch's "convolution"
compute, what is the classical name for it, and which NumPy function
must you therefore *not* reconcile against?

**Q4.** Receptive field of the stack ($5\times5$ s1, $3\times3$ s2,
$3\times3$ s1), by the growth rule — show $r$ and $J$ after each
layer.

**Q5.** Give the four-word depth ordering of the feature hierarchy
and the two map statistics that separate early from late, with their
directions.

**Q6.** The three reproducibility habits when loading a pretrained
model, and the specific failure each prevents.

**Q7.** ResNet-50's stage shape table for `(B, 3, 224, 224)`: channels
and spatial size after `maxpool`, `layer1`, `layer2`, `layer3`,
`layer4`, `avgpool`.

**Q8.** Why does `nn.Sequential(*list(model.children()))` crash where
`model(x)` succeeds — and which single module, inserted where, fixes
the rebuild?

**Q9.** Hand-count the three convolutions of a `layer4` interior
bottleneck ($2048 \to 512 \to 512 \to 2048$), then its BN
parameters, then the block total.

**Q10.** Which blocks of ResNet-50 carry a `downsample`, which of
those halve the grid, and what two jobs can a downsample do?

**Q11.** The trunk `nn.Sequential(*ch[:5], model.layer3[:1])` — is it
valid, and if not, why exactly (state the shape that arrives and the
shape that is expected)?

**Q12.** Freeze-then-graft versus graft-then-freeze on a deepcopied
model with `fc = nn.Linear(2048, 6)`: the `n_trainable` audit value
each order produces.

**Q13.** A transfer build on layer2 features ($f = 512$) for $k = 25$
classes: frozen scalars, trainable scalars, output shape — all by
hand.

**Q14.** Your build's audit prints trainable names
`['head.weight', 'head.bias', 'backbone.7.2.bn3.weight',
'backbone.7.2.bn3.bias']`.
Diagnose: what went wrong, why might the *count* audit have looked
plausible, and what loop repairs it?



## What to redo, per weak spot

- **Convolution arithmetic, component form (Q1–Q3):** redo p01, p02,
  p06; reread Session 1 §§1–5 and Pitfall 1.
- **Feature maps and banks (Q2, Q3):** redo p07, p21; Session 1 §6.
- **Receptive fields (Q4):** redo p05, p13, p20; Session 1 §7.
- **Feature hierarchy (Q5):** redo p03, p18, p22; Session 1 §8.
- **Loading discipline and the float32 register (Q6):** redo p12;
  Session 2 §1 and Pitfall 2.
- **ResNet anatomy and shapes (Q7, Q8):** redo p08, p12, p15;
  Session 2 §§2–4.
- **Parameter arithmetic (Q9):** redo p04, p14, p15; Session 2
  §§5–6 — especially the count-without-`numel` register.
- **Truncation (Q10, Q11):** redo p09, p16; Session 3 §§2–3.
- **Freezing and audits (Q12):** redo p10, p16; Session 3 §4 and
  Pitfall 1.
- **Transfer builds (Q13, Q14):** redo p11, p17, p19; Session 3
  §§5–7.
- **The end-to-end texture:** if any hesitation remains, rebuild
  Session 3 §6's `TransferNet` from a blank cell — it is the unit in
  one construction.



## Answers (open only when done)

<details><summary><b>Answers to all 14 quiz items</b></summary>

**A1.** Length $5 - 2 + 1 = 4$:
$(1\cdot2 + 2\cdot1,\; 2\cdot2 - 1\cdot1,\; -1\cdot2 + 0\cdot1,\;
0\cdot2 + 3\cdot1) = (4, 3, -2, 3)$.

**A2.** Output $(4, 10, 32, 32)$ — $\lfloor(63 + 4 - 5)/2\rfloor + 1
= 32$; weight $(10, 3, 5, 5)$; parameters
$10\cdot3\cdot25 + 10 = 760$.

**A3.** Torch slides the kernel as written — no reversal — which is
classically *cross-correlation*; `np.convolve` flips, so
reconciliations must use the course's component form, not
`np.convolve`.

**A4.** Start $r = 1, J = 1$.
$5\times5$ s1: $r = 5$, $J = 1$.
$3\times3$ s2: $r = 5 + 2 = 7$, then $J = 2$.
$3\times3$ s1: $r = 7 + 2\cdot2 = 11$, $J = 2$.
RF $= 11$.

**A5.** Edges → textures → parts → objects.
Activation fraction: high early, low late (late maps are sparse);
roughness: high early, low late (late maps vary slowly).

**A6.** Explicit `weights=ResNet50_Weights.IMAGENET1K_V1` (pins the
artifact); `model.eval()` (stops batch-statistics BatchNorm:
nondeterminism across batch composition + buffer mutation);
`torch.inference_mode()` around every forward (stops silent autograd
graph building — pretrained flags arrive `True`).

**A7.** `maxpool` $(B, 64, 56, 56)$; `layer1` $(B, 256, 56, 56)$;
`layer2` $(B, 512, 28, 28)$; `layer3` $(B, 1024, 14, 14)$;
`layer4` $(B, 2048, 7, 7)$; `avgpool` $(B, 2048, 1, 1)$.

**A8.** The model's `forward` calls `torch.flatten(x, 1)` between
`avgpool` and `fc`; `children()` contains no module that does this,
so `fc` receives $(B, 2048, 1, 1)$ and the matmul fails.
Insert `nn.Flatten(1)` between `avgpool` and `fc` in the rebuilt
list.

**A9.** Convs: $512\cdot2048 + 512\cdot512\cdot9 + 2048\cdot512 =
1{,}048{,}576 + 2{,}359{,}296 + 1{,}048{,}576 = 4{,}456{,}448$
($= 17m^2$ at $m = 512$).
BN: $2(512 + 512 + 2048) = 6{,}144$ ($= 12m$).
Total $4{,}462{,}592$.

**A10.** Block 0 of every stage.
`layer2[0]`, `layer3[0]`, `layer4[0]` halve the grid;
`layer1[0]`'s downsample is stride 1.
The two jobs: match the skip's *channel count* to the block output,
and (when stride 2) match the halved *grid*.

**A11.** Invalid: `ch[:5]` ends at `layer1`, producing
$(B, 256, 56, 56)$, but `layer3[0]` expects `layer2`'s output
interface $(B, 512, 28, 28)$ — its 1×1 reduce reads 512 channels.
The forward crashes at that first conv.

**A12.** Freeze-then-graft: $2049 \cdot 6 = 12{,}294$ trainable
(the fresh head, born `True`).
Graft-then-freeze: $0$ — the loop swallowed the head.

**A13.** Frozen $= 9{,}408 + 128 + 215{,}808 + 1{,}219{,}584 =
1{,}444{,}928$; trainable $= 512\cdot25 + 25 = 12{,}825$; output
$(B, 25)$.

**A14.** A selective freeze missed one backbone BatchNorm — the name
`backbone.7.2.bn3` reads: child 7 of the slice (`layer4`), block 2,
final BN — so two backbone tensors remain trainable (bad prefix
filter or a thaw leak).
The scalar count barely moves ($+2C$ on a six-figure expectation
could pass a sloppy "looks close" check) — which is exactly why the
*name* audit exists: membership must be `{head.weight, head.bias}`.
Repair:
`for n, p in net.named_parameters():
p.requires_grad = n.startswith("head")` — or rerun the backbone-wide
freeze loop before grafting.

</details>
